In [1]:
import os
from collections import defaultdict

import h5py
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader


class SegmentFeatureDataset(Dataset):
    """
    One item = one EEG segment.

    Returns:
        {
            "labram": Tensor,
            "welch": Tensor,
            "eeg_id": str,
            "segment_idx": int,
            "n_segments": int,
            "start_sec": float,
            "end_sec": float,
        }
    """
    def __init__(
        self,
        root_dir: str,
        index_file: str = "index.parquet",
        use_labram: bool = True,
        use_welch: bool = True,
        transform=None,
    ):
        self.root_dir = root_dir
        self.index_path = os.path.join(root_dir, index_file)
        self.df = pd.read_parquet(self.index_path).reset_index(drop=True)

        self.use_labram = use_labram
        self.use_welch = use_welch
        self.transform = transform

        self._h5_cache = {}

    def __len__(self):
        return len(self.df)

    def _get_h5(self, shard_name: str):
        if shard_name not in self._h5_cache:
            shard_path = os.path.join(self.root_dir, shard_name)
            self._h5_cache[shard_name] = h5py.File(shard_path, "r")
        return self._h5_cache[shard_name]

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        h5f = self._get_h5(row["shard"])
        shard_row = int(row["row"])

        out = {
            "eeg_id": str(row["eeg_id"]),
            "segment_idx": int(row["segment_idx"]),
            "n_segments": int(row["n_segments"]),
            "start_sec": float(row["start_sec"]),
            "end_sec": float(row["end_sec"]),
        }

        if self.use_labram:
            labram = h5f["LaBraM"][shard_row]
            out["labram"] = torch.tensor(labram, dtype=torch.float32)

        if self.use_welch:
            welch = h5f["Welch"][shard_row]
            out["welch"] = torch.tensor(welch, dtype=torch.float32)

        if self.transform is not None:
            out = self.transform(out)

        return out

    def close(self):
        for f in self._h5_cache.values():
            try:
                f.close()
            except Exception:
                pass
        self._h5_cache = {}

    def __del__(self):
        self.close()


class EEGFeatureDataset(Dataset):
    """
    One item = one EEG, containing all of its segments.

    Returns:
        {
            "labram": Tensor [N_seg, ...]   (optional)
            "welch": Tensor [N_seg, ...]    (optional)
            "eeg_id": str
            "segment_idx": Tensor [N_seg]
            "start_sec": Tensor [N_seg]
            "end_sec": Tensor [N_seg]
        }
    """
    def __init__(
        self,
        root_dir: str,
        index_file: str = "index.parquet",
        use_labram: bool = True,
        use_welch: bool = True,
    ):
        self.root_dir = root_dir
        self.index_path = os.path.join(root_dir, index_file)
        self.df = pd.read_parquet(self.index_path).reset_index(drop=True)

        self.use_labram = use_labram
        self.use_welch = use_welch

        self._h5_cache = {}

        # group rows by eeg_id
        self.eeg_ids = []
        self.groups = []

        for eeg_id, g in self.df.groupby("eeg_id", sort=False):
            g = g.sort_values("segment_idx").reset_index(drop=True)
            self.eeg_ids.append(str(eeg_id))
            self.groups.append(g)

    def __len__(self):
        return len(self.groups)

    def _get_h5(self, shard_name: str):
        if shard_name not in self._h5_cache:
            shard_path = os.path.join(self.root_dir, shard_name)
            self._h5_cache[shard_name] = h5py.File(shard_path, "r")
        return self._h5_cache[shard_name]

    def __getitem__(self, idx: int):
        g = self.groups[idx]
        eeg_id = self.eeg_ids[idx]

        out = {
            "eeg_id": eeg_id,
            "segment_idx": torch.tensor(g["segment_idx"].to_numpy(), dtype=torch.long),
            "start_sec": torch.tensor(g["start_sec"].to_numpy(), dtype=torch.float32),
            "end_sec": torch.tensor(g["end_sec"].to_numpy(), dtype=torch.float32),
        }

        if self.use_labram:
            labram_list = []
            for _, row in g.iterrows():
                h5f = self._get_h5(row["shard"])
                labram_list.append(h5f["LaBraM"][int(row["row"])])
            out["labram"] = torch.tensor(np.stack(labram_list), dtype=torch.float32)

        if self.use_welch:
            welch_list = []
            for _, row in g.iterrows():
                h5f = self._get_h5(row["shard"])
                welch_list.append(h5f["Welch"][int(row["row"])])
            out["welch"] = torch.tensor(np.stack(welch_list), dtype=torch.float32)

        return out

    def close(self):
        for f in self._h5_cache.values():
            try:
                f.close()
            except Exception:
                pass
        self._h5_cache = {}

    def __del__(self):
        self.close()

In [2]:
DATA_DIR = r"H:\EEG_features\EEG_features_labram_welch_HV"

test_Dataset = EEGFeatureDataset(root_dir=DATA_DIR)

In [15]:
test_Dataset[10000]['labram'].shape

torch.Size([30, 200])

In [16]:
import mne

# read fif

raw = mne.io.read_raw_fif(r"H:\EEG\FHA\hyperventilation\Burnaby\844b7325-6ffe-4860-ad9d-98f14cda8002_raw.fif", preload=True)

Opening raw data file H:\EEG\FHA\hyperventilation\Burnaby\844b7325-6ffe-4860-ad9d-98f14cda8002_raw.fif...
    Range : 0 ... 15615 =      0.000 ...    60.996 secs
Ready.
Reading 0 ... 15615  =      0.000 ...    60.996 secs...


In [18]:
raw.get_data().shape

(24, 15616)